In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import astropy.constants as c
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline, RectBivariateSpline
from matplotlib import cm, colors
from matplotlib.ticker import LogLocator, FuncFormatter
import niceplots.utils as nicepl


nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
Cp

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
    "Smooth_resolution": False,
    # "nonlinearMatpow": False,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

In [ ]:
def get_specs(z, nu):
        nuObs = nu / (z + 1)
        
        spectrograph_R = 100
        l = c.c / nuObs
        dl = l / spectrograph_R
        dnu = (c.c / dl).to(nuObs.unit)

        dz = 0.5
        ld = l * (z - dz/2)
        lu = l * (z + dz/2)
        Deltanu = (c.c / ld - c.c / lu).to(nuObs.unit)

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 33 * u.arcsec,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": Deltanu, # Frequency Bin
                "dnu": dnu, # Spectrograph resolution
                "Omega_field": 18.64 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

# Study the Response Functions for Different Lines

In [ ]:
astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
}
nu = 1.897 * u.THz
surveyspecs_CII = get_specs(np.array([1]), nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

In [ ]:
pobs_CII = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    astrodict_CII,
    surveyspecs_CII,
    pobs_settings={"mu_kind":"gauss", "kmin":5e-3 * u.Mpc**-1, "nk":200},
    output=["Power spectrum"],
)["Power spectrum"]
ssc_CII = scov.SuperSampleCovariance(pobs_CII)

In [ ]:
mycosmo = pobs_CII.cosmology
myastro = pobs_CII.astro
k = pobs_CII.k

In [ ]:
from time import time

In [ ]:
z = np.linspace(0, 9)
t=time()
for x in range(150):
    mycosmo.Pk_l.f(1e-4 * u.Mpc**-1, z)
print(time()-t)

t=time()
for x in range(150):
    mycosmo.growth_rate(1e-4 * u.Mpc**-1, z)
print(time()-t)

In [ ]:
I21 = myastro.Thalo(1, k, p=1, scale=(2,), beta=1)
I40 = myastro.Thalo(1, k, p=1, scale=(4,), beta=0)
(I21**2).unit

In [ ]:
I40.unit

In [ ]:
myastro.halomodel.Ihalo(1, 1e-3*u.Mpc**-1, p=1, beta=0)

In [ ]:
color = iter(Cs)

plt.semilogx(k, (I21/I21[0])**2, c=next(color), label=r"$\left[\mathcal{I}_1^2\right]^2$")
plt.semilogx(k, I40/I40[0], c=next(color), ls="--", label=r"$\mathcal{I}^4_0$")
plt.ylabel(r"$\mathcal{I}(k) / \mathcal{I}(0)$")
plt.xlabel(r"Wavenumber $k$")
plt.legend()
plt.title("[CII], $z=1$")

In [ ]:
M = myastro.M
int21 = myastro.massluminosityfunction(M, 1)**2 * myastro.halomodel.halomassfunction(M, 1) * M * myastro.halomodel.get_bias(M, 1, 1)
int40 = myastro.massluminosityfunction(M, 1)**4 * myastro.halomodel.halomassfunction(M, 1) * M


In [ ]:
color = iter(Cs)

plt.semilogx(M, int21 / int21.max(), c=next(color), label=r"$\mathcal{I}_1^2$")
plt.semilogx(M, int40 / int40.max(), c=next(color), ls="--", label=r"$\mathcal{I}^4_0$")

plt.ylabel(r"normalised Integrand of $\mathcal{I}$")
plt.xlabel(r"Halomass $M\,[M_\odot]$")
plt.legend()
plt.title("[CII], $z=1$")

In [ ]:
color = iter(Cp)
from itertools import product

alpha = [1, 2]
for b, ai in product(zip([1,2,3],[1, "b2_fitted", "b3"]), alpha):
    inta1 = myastro.massluminosityfunction(M, 1)**ai * myastro.halomodel.halomassfunction(M, 1) * M * myastro.halomodel.get_bias(M, 1, b[1])

    norm = np.abs(inta1).max() * np.sign(inta1[np.argmax(np.abs(inta1))])

    plt.semilogx(M, inta1 / norm, c=next(color), label=r"$\mathcal{{I}}_{}^{}$".format(b[0], ai))

plt.ylabel(r"normalised Integrand of $\mathcal{I}$")
plt.xlabel(r"Halomass $M\,[M_\odot]$")
plt.legend()
plt.title("[CII], $z=1$")

In [ ]:
color = iter(Cs)

LiG = ssc_CII.linear_growth_response    (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
BiC = ssc_CII.biased_clustering_response(pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
HSV = ssc_CII.halo_sample_variance      (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
plt.semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
plt.semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
plt.semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
plt.semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
plt.title("[CII], $z=1$")

In [ ]:
astrodict_CO21={
    "model_type": "ML",
    "model_name": "TonyLi",
    "model_par": {
        "alpha": 1.11,
        "beta": 0.6,
        "dMF": 1 * u.Msun * u.yr**-1 * u.Lsun**-1,
        "sig_SFR":0,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
} # CO(2-1)

nu = 2 * 115.27 * u.GHz # CO(2-1)
surveyspecs_CO21 = get_specs(np.array([1]),nu)

In [ ]:
pobs_CO21 = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    astrodict_CO21,
    surveyspecs_CO21,
    pobs_settings={"mu_kind":"gauss", "kmin":5e-3 * u.Mpc**-1},
    output=["Power spectrum"],
)["Power spectrum"]
ssc_CO21 = scov.SuperSampleCovariance(pobs_CO21)

In [ ]:
color = iter(Cs)

LiG = ssc_CO21.linear_growth_response    (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
BiC = ssc_CO21.biased_clustering_response(pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
HSV = ssc_CO21.halo_sample_variance      (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
plt.semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
plt.semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
plt.semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
plt.semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
plt.title("CO(2-1), $z=1$")

In [ ]:
fw, fh=plt.rcParams['figure.figsize']
fig, axs=plt.subplots(1,2, figsize=(2*fw, 1.1*fh), sharex=True, sharey=True) 

color = iter(Cs)
LiG = ssc_CII.linear_growth_response    (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
BiC = ssc_CII.biased_clustering_response(pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
HSV = ssc_CII.halo_sample_variance      (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
axs[0].semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
axs[0].semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
axs[0].semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
axs[0].semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
axs[0].set_xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
axs[0].set_ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
axs[0].set_title("[CII], $z=1$")

color = iter(Cs)
LiG = ssc_CO21.linear_growth_response    (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
BiC = ssc_CO21.biased_clustering_response(pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
HSV = ssc_CO21.halo_sample_variance      (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
axs[1].semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
axs[1].semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
axs[1].semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
axs[1].semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
axs[1].set_xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
axs[1].set_ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
axs[1].set_title("CO(2-1), $z=1$")

handles, labels = [], []

for ax in [axs[0]]:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)

fig.legend(handles, labels, loc='upper center', ncol=4)
plt.tight_layout(rect=[0, 0, 1, 0.93])

Neutral Hydrogen

In [ ]:
astrodict_HI={
    "model_type": "ML",
    "model_name": "L_from_MHI_VN",
    "model_par": {
        "alpha": 0.53,
        "M0": 1.5e10 * myssl.current_astro.Msunh,
        "Mmin":6.0e11 * myssl.current_astro.Msunh,
        "do_quench": False,
    },
    "sigma_scatter" : 0.0,
}
nu = (c.c/ (21*u.cm)).to(u.MHz)
surveyspecs_HI = get_specs(np.array([1]), nu)

In [ ]:
(1.5e10 * myssl.current_astro.Msunh * 6.25e-9 * u.Lsun / u.Msun).to(u.Lsun)

In [ ]:
pobs_HI = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    astrodict_HI,
    surveyspecs_HI,
    pobs_settings={"mu_kind":"gauss", "kmin":5e-3 * u.Mpc**-1},
    output=["Power spectrum"],
)["Power spectrum"]
ssc_HI = scov.SuperSampleCovariance(pobs_HI)

In [ ]:
fw, fh=plt.rcParams['figure.figsize']
fig, axs=plt.subplots(1,2, figsize=(2*fw, 1.1*fh), sharex=True, sharey=True) 

color = iter(Cs)
LiG = ssc_CII.linear_growth_response    (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
BiC = ssc_CII.biased_clustering_response(pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
HSV = ssc_CII.halo_sample_variance      (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
axs[0].semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
axs[0].semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
axs[0].semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
axs[0].semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
axs[0].set_xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
axs[0].set_ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
axs[0].set_title("[CII], $z=1$")

color = iter(Cs)
LiG = ssc_HI.linear_growth_response    (pobs_HI.k, pobs_HI.z) / pobs_HI.Pk_0bs
BiC = ssc_HI.biased_clustering_response(pobs_HI.k, pobs_HI.z) / pobs_HI.Pk_0bs
HSV = ssc_HI.halo_sample_variance      (pobs_HI.k, pobs_HI.z) / pobs_HI.Pk_0bs
axs[1].semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
axs[1].semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
axs[1].semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
axs[1].semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
axs[1].set_xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
axs[1].set_ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
axs[1].set_title("HI, $z=1$")

handles, labels = [], []

for ax in [axs[0]]:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)

fig.legend(handles, labels, loc='upper center', ncol=4)
plt.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig("output/CII_vs_HI_logResponse.pdf")

# Vary Omega_field and look for covariances

In [ ]:
def get_specs(z, nu, Omf):
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.10

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": Omf * u.deg**2,
        }
        return surveyspecs

In [ ]:
k_test = np.geomspace(1e-3, 1, 200) * u.Mpc**-1

In [ ]:
ois = 1.4 * np.geomspace(1, 10000, 5)
list_specs = [get_specs(1, 1.897 * u.THz, oi) for oi in ois]



In [ ]:
Vsurvey_arr = []
integrand = []
sigmaV_arr = []
SSC_arr = []

for specs in list_specs:
        pobs_CII = myssl.compute(
                myssl.fiducialcosmoparams,
                myssl.fiducialhaloparams,
                astrodict_CII, specs,
                pobs_settings={"mu_kind":"linear", "kmin":5e-3 * u.Mpc**-1, "nk":50},
                output=["Power spectrum"])["Power spectrum"]
        kcov = pobs_CII.k
        V = pobs_CII.survey_specs.Vfield()
        Vsurvey_arr.append(V)

        ssc_obj = scov.SuperSampleCovariance(pobs_CII)
        t, I = ssc_obj.sigma_survey_intg(alpha=3)
        integrand.append(I)
        sigmaV_arr.append(ssc_obj.sigma_survey())
        SSC_arr.append(ssc_obj.compute_SSC().squeeze())

Vsurvey_arr = np.array([V.value for V in Vsurvey_arr])
integrand = np.array(integrand)
sigmaV_arr = np.array(sigmaV_arr)
SSC_arr = np.array(SSC_arr)

In [ ]:
C = seaborn.color_palette("viridis", 5)
C

In [ ]:
color = iter(C)
for I in integrand.squeeze():
    plt.semilogy(t, I, c=next(color))

In [ ]:
unit = Vsurvey_arr[0].unit
Vsurvey_arr = np.array(
    [Vs.to(unit).value for Vs in Vsurvey_arr]
) * unit
Vsurvey_arr

In [ ]:
plt.loglog(ois, sigmaV_arr * Vsurvey_arr)

In [ ]:
pobs_CII.survey_specs.get_redshifts()

In [ ]:
Gcov_arr = scov.Covariance(pobs_CII).gaussian_nonoise_cov()[:, 0, 0, 0]
NGcov_arr = scov.nonGuassianCov(pobs_CII).compute_nG_Cov()[..., 0]

In [ ]:
fw, fh = plt.rcParams['figure.figsize']
fig, ax = plt.subplots(1, 1, figsize=(fw, 1.3*fh))

for i in range(len(ois)):
    ax.loglog(kcov.to(pobs_CII.astro.Mpch**-1), Vsurvey_arr[i] * np.diag(SScov_arr[i, ...]),
              c=Cs[2], alpha=1 - i / (len(ois)+1))
    ax.loglog([],[], c="gray", alpha=1 - i / (len(ois)+1), label=r"$\Omega_\mathrm{field}="+f"{ois[i]}\,\mathrm{{deg}}^2$") 

ax.loglog(kcov.to(pobs_CII.astro.Mpch**-1), Vsurvey_arr[i] * Gcov_arr,
            c=Cs[0])
ax.loglog(kcov.to(pobs_CII.astro.Mpch**-1), Vsurvey_arr[i] * np.diag(NGcov_arr),
            c=Cs[1])
ax.loglog(kcov.to(pobs_CII.astro.Mpch**-1), Vsurvey_arr[i] * -np.diag(NGcov_arr),
            c=Cs[1], ls="--")


ax.set_xlabel(r"$k_1\,[h\,\mathrm{Mpc}^{-1}]$")
ax.set_ylabel(r"$V_\mathrm{eff}\times\mathrm{Cov}(k_1,k_1)\,[\mu\mathrm{K}^{4}]$")
ax.set_title("[CII], $z=1$")

# First legend (field size)
legend1 = ax.legend(loc="lower left")

# Create dummy lines for covariance types
line_G, = ax.plot([], [], color=Cs[0], label="Gaussian")
line_NG, = ax.plot([], [], color=Cs[1], label="Non-Gaussian")
line_SS, = ax.plot([], [], color=Cs[2], label="Super-sample")

# Second legend (covariance contributions)
legend2 = ax.legend(handles=[line_G, line_NG, line_SS], loc="upper right")

# Add the first legend back
ax.add_artist(legend1)

plt.show()

In [ ]:
pobs_CII.survey_specs.get_redshifts()

# Reproduce Concerto Results

In [ ]:
mycosmo = pobs_CII.fiducial_cosmology

In [ ]:
import multiprocessing as mp
from functools import partial
from scipy.special import spherical_jn
from scipy.integrate import dblquad

In [ ]:
from scipy.integrate import lebedev_rule
points, weights = lebedev_rule(131)
kx, ky, kz = (*points,)

In [ ]:
def A_of_u(k, La, Lb):
        ka = (k * La).to(1).value[:, None]
        kb = (k * Lb).to(1).value[:, None]

        Jx = spherical_jn(0, 0.5*ka*kx)
        Jy = spherical_jn(0, 0.5*ka*ky)
        Jz = spherical_jn(0, 0.5*kb*kz)

        return np.sum(weights * (Jx * Jy * Jz)**2, axis=1) / (4 * np.pi)

def get_specs(nuObs, Omf):
        nu = 1.897 * u.THz

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": 10 * u.GHz, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": Omf,
        }
        return surveyspecs
nus = np.geomspace(125, 305, 15) * u.GHz
ois = np.array([0.0625, 0.25, 1, 4, 9]) * u.deg**2
list_specs = [get_specs(nui, oi) for nui, oi in product(nus, ois)]

sigma_over_mu = []
for specs in list_specs:
        pobs_CII = myssl.compute(
                myssl.fiducialcosmoparams,
                myssl.fiducialhaloparams,
                astrodict_CII, specs,
                pobs_settings={"mu_kind":"gauss", "kmin":5e-3 * u.Mpc**-1, "nk":50},
                output=["Power spectrum"])["Power spectrum"]

        sscov = scov.SuperSampleCovariance(pobs_CII)
        sigma_SSC = np.sqrt(
                np.diag(
                        sscov.compute_SSC().squeeze()
                )
        )
        sigma_over_mu.append(sigma_SSC / pobs_CII.Pk_0bs.squeeze())
sigma_over_mu = np.array(sigma_over_mu)

In [ ]:
sigma_over_mu.reshape((15, 5, 49))
fmask = np.any(~np.isnan(sigma_over_mu), axis=-1)

In [ ]:
sigma_over_mu[fmask].shape

In [ ]:
pobs_CII.Pk_0bs.squeeze().shape

In [ ]:
def concerto(nu, dnu, om, c, alpha, beta, gamma):
    A = (nu / (100 * u.GHz)).to(1).value
    B = (dnu / (5 * u.GHz)).to(1).value
    C = (om / (1 * u.deg**2)).to(1).value
    return c * A**alpha * B**beta * C**gamma

In [ ]:
c = concerto(nus[9:], 10* u.GHz, ois[:, None], 6.22, -2.4, -0.61, -0.46).T

In [ ]:
pobs_CII.k.shape

In [ ]:
list = [[nui.value, oi.value] for nui, oi in product(nus, ois)]
np.array(list).reshape((2, 5, 15))[0, 0, :]

In [ ]:
poision = sigma_over_mu[fmask, :]
poision.reshape(15, 5, 49)

In [ ]:
color = iter(Cp)
for i in range(6):
    plt.scatter(ois, poision[i, :], color=next(color))
    plt.loglog(ois,  c[i, :], color=next(color))

In [ ]:
color = iter(Cp)
for i in range(5):
    plt.scatter(nus[9:].value, poision[:, i], color=next(color))
    plt.loglog( nus[9:].value,  c[:, i], color=next(color))

plt.loglog([],[], color="grey", label="Concerto Fit")
plt.scatter([],[], color="lightgrey", label="SSC model")
plt.legend()

plt.xlabel(r"$\nu_{\rm obs}\,[\mathrm{GHz}]$")
plt.ylabel(r"Relative Error")

In [ ]:
W = specs_obj.Wsurvey(k, pobs_CII.mu) / specs_obj.Vfield()
W = np.sum(pobs_CII.w * (specs_obj.Wsurvey(k, pobs_CII.mu) / specs_obj.Vfield())**2, axis=1)


In [ ]:
plt.loglog(k, W)